# Task 1 — Loading, Inspecting and Cleaning the Grid Datasets

Three CSVs, generated by the seeded script in `../scripts/generate_grid_data.py`:

| file | one row is... | modelled on |
|---|---|---|
| `utilities.csv` | a power company (ECG, GRIDCo, ...) | airlines.csv |
| `substations.csv` | a substation with location, voltage, capacity | airports.csv |
| `lines.csv` | a transmission/distribution line between two substations | routes.csv |

The generator is seeded with 42, so every run and every team gets identical files.
That matters: if our numbers differ from another team's, someone's pipeline is wrong,
not their data.

The cleaning logic itself lives in `../src/griddata.py` so the other notebooks can
reuse it without copy-pasting. This notebook walks through what it does and shows the
evidence.

In [1]:
import sys

sys.path.append("../src")

import pandas as pd

import griddata

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 25)

raw = griddata.load_raw()

## 1. First look at the raw files

Before touching anything: what did we actually get? `.info()` shows column types and
non-null counts, `.head()` shows what the values look like in practice.

In [2]:
for name, df in raw.items():
    print(f"===== {name} — {df.shape[0]} rows x {df.shape[1]} cols =====")
    df.info()
    print()

===== utilities — 10 rows x 7 cols =====
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Utility ID  10 non-null     int64
 1   Name        10 non-null     str  
 2   Alias       10 non-null     str  
 3   Code        10 non-null     str  
 4   Type        10 non-null     str  
 5   Country     10 non-null     str  
 6   Active      10 non-null     str  
dtypes: int64(1), str(6)
memory usage: 1.2 KB

===== substations — 44 rows x 12 cols =====
<class 'pandas.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Substation ID       44 non-null     int64  
 1   Name                44 non-null     str    
 2   Short Name          44 non-null     str    
 3   Region              44 non-null     str    
 4   Country             44 non-null     

In [3]:
raw["utilities"].head()

,Utility ID,Name,Alias,Code,Type,Country,Active
0,1,Electricity Company of Ghana,ECG,ECG,Distribution,Ghana,Y
1,2,Northern Electricity Distribution Company,NEDCo,NED,Distribution,Ghana,Y
2,3,Ghana Grid Company,GRIDCo,GRD,Transmission,Ghana,Y
3,4,Volta River Authority,VRA,VRA,Generation,Ghana,Y
4,5,Compagnie Ivoirienne d'Electricite,CIE,CIE,Distribution,Cote d'Ivoire,Y


In [4]:
raw["substations"].head()

,Substation ID,Name,Short Name,Region,Country,Latitude,Longitude,Voltage (kV),Capacity (MVA),Commissioning Year,Type,Status
0,1,Achimota Substation,Achimota,Greater Accra,Ghana,5.6085,-0.2193,11,6.4,2008,Distribution,Active
1,2,Tema Substation,Tema,Greater Accra,Ghana,5.6596,-0.0226,330,48.5,1997,Transmission,Active
2,3,Mallam Substation,Mallam,Greater Accra,Ghana,5.5630,-0.2971,330,25.2,1979,Transmission,Active
3,4,Legon Substation,Legon,Greater Accra,Ghana,5.6401,-0.1799,161,241.9,2009,Transmission,Active
4,5,Kaneshie Substation,Kaneshie,Greater Accra,Ghana,5.5691,-0.2413,161,146.0,1970,Transmission,Active


In [5]:
raw["lines"].head()

,Line ID,Utility ID,Source Substation ID,Source Substation,Destination Substation ID,Destination Substation,Voltage (kV),Length (km),Capacity (MVA),Status,Line Type
0,1,2,1,Achimota Substation,2,Tema Substation,11,25.2,215.0,Active,Underground
1,2,2,1,Achimota Substation,3,Mallam Substation,11,11.0,36.4,Active,Overhead
2,3,3,1,Achimota Substation,5,Kaneshie Substation,11,5.9,277.4,Active,Overhead
3,4,1,2,Tema Substation,3,Mallam Substation,330,34.4,39.0,Active,Underground
4,5,3,2,Tema Substation,4,Legon Substation,161,19.5,186.5,Active,Underground


Observations from the raw load:

- Every column pandas should read as numeric already is (Latitude, Longitude,
  Capacity...). There are no `\N` markers or blank strings — unlike the raw
  OpenFlights files this dataset is modelled on, the generator writes clean CSVs.
- No nulls anywhere, per the non-null counts above.
- Column names contain spaces and units (`Voltage (kV)`), so they need quoting in
  every `df[...]` — mildly annoying but not worth renaming and then having every
  figure caption differ from the CSV headers.

## 2. Cleaning

Even though the generated data is clean, real asset registers never are — so the
pipeline treats this file the way it would treat one exported from a utility's
twenty-year-old asset database. Each step logs what it changed so the report can
state exactly what was done. The steps:

1. Replace the usual "missing" placeholders (`\N`, `NULL`, empty strings, `-`) with
   real NaN so pandas sees them as missing.
2. Strip leading/trailing whitespace from every text value — `" Tema"` and `"Tema"`
   must group together.
3. Coerce numeric columns with `errors='coerce'`, so garbage like `"unknown"` in a
   capacity column becomes NaN instead of crashing the pipeline.
4. Drop exact duplicate rows.
5. Count what's still missing. We deliberately do **not** impute: inventing a
   capacity value for a substation would contaminate the capacity statistics later,
   and with this dataset there is nothing missing to impute anyway.

In [6]:
data, cleaning_report = griddata.clean(raw)
cleaning_report

,Table,Step,Result
0,utilities,Strip whitespace from text columns,0 value(s) changed
1,utilities,Coerce numeric columns,0 value(s) became NaN
2,utilities,Drop duplicate rows,0 removed
3,utilities,Remaining missing values,0 cell(s)
4,substations,Strip whitespace from text columns,0 value(s) changed
5,substations,Coerce numeric columns,0 value(s) became NaN
6,substations,Drop duplicate rows,0 removed
7,substations,Remaining missing values,0 cell(s)
8,lines,Strip whitespace from text columns,0 value(s) changed
9,lines,Coerce numeric columns,0 value(s) became NaN


All zeros — expected for generated data, but now it's demonstrated rather than
assumed, and the same pipeline would catch real problems in a real file.

## 3. Validation

Cleaning fixes formats; validation checks the *relationships*. The ones that matter
here:

- **Primary keys unique** — duplicate IDs would corrupt every join.
- **Foreign keys resolve** — a line whose source substation doesn't exist would appear
  in the network graph as a phantom node with no location or capacity.
- **Coordinates inside West Africa** — a transposed lat/lon puts a substation in the
  ocean and wrecks the maps and every distance calculation.
- **Categorical fields in their expected sets** — a typo like `"Actve"` would silently
  drop rows from every status filter.
- **Sanity**: no negative line lengths, no self-loops, no future commissioning years.

In [7]:
validation = griddata.validate(data)
validation

,severity,check,detail,count
0,OK,utilities.Utility ID unique,0 duplicate id(s),0
1,OK,substations.Substation ID unique,0 duplicate id(s),0
2,OK,lines.Line ID unique,0 duplicate id(s),0
3,OK,lines.Source Substation ID resolves to a subst...,0 orphaned reference(s),0
4,OK,lines.Destination Substation ID resolves to a ...,0 orphaned reference(s),0
5,OK,lines.Utility ID resolves to a utility,0 orphaned reference(s),0
6,OK,Coordinates inside West Africa,"0 substation(s) outside (3.0, 16.0) / (-18.0, ...",0
7,OK,substations.Voltage (kV) in expected set,all values expected,0
8,OK,substations.Type in expected set,all values expected,0
9,OK,substations.Status in expected set,all values expected,0


In [8]:
errors = validation[validation["severity"] == "ERROR"]
warns = validation[validation["severity"] == "WARN"]
print(f"errors: {len(errors)}, warnings: {len(warns)}")
assert errors.empty, "fix data errors before continuing to any analysis"

errors: 0, warnings: 1


### The two warnings are worth understanding

**Two substations have no lines at all** — Conakry Transmission Hub and Savelugu
Substation. The brief's own sample run reports the graph as "42 nodes, 55 edges"
because building a graph straight from the edge list never sees a node with no edges.
That is exactly the kind of quiet data loss the validation step exists to surface:
the network has 44 substations, two of which are unreachable — itself a finding about
grid coverage (Conakry is a WAPP interconnection point that the generator's line
logic never connected; Savelugu is a Northern-region distribution substation).

We keep both in the dataset and add them to the graph explicitly as isolated nodes.
Dropping them would misreport substation counts per region, total capacity, and
asset-age statistics.

## 4. Basic statistics for each dataset

In [9]:
griddata.summarise(data)

Utilities                    10.0
Substations                  44.0
Lines                        55.0
Regions                      18.0
Countries                     6.0
Active substations           43.0
Lines under maintenance       2.0
Total capacity (MVA)       6946.1
Total line length (km)     5462.2
Oldest substation          1967.0
Newest substation          2022.0
dtype: float64

In [10]:
data["substations"][["Voltage (kV)", "Capacity (MVA)", "Commissioning Year"]].describe().round(1)

,Voltage (kV),Capacity (MVA),Commissioning Year
count,44.0,44.0,44.0
mean,134.5,157.9,1996.3
std,120.4,139.9,16.1
min,11.0,6.4,1967.0
25%,33.0,43.8,1982.2
50%,69.0,108.6,1999.5
75%,161.0,254.4,2009.2
max,330.0,487.6,2022.0


In [11]:
data["lines"][["Voltage (kV)", "Length (km)", "Capacity (MVA)"]].describe().round(1)

,Voltage (kV),Length (km),Capacity (MVA)
count,55.0,55.0,55.0
mean,141.4,99.3,222.4
std,135.2,90.3,109.2
min,11.0,3.8,32.9
25%,22.0,42.9,134.6
50%,69.0,75.9,229.9
75%,330.0,129.6,292.1
max,330.0,426.0,506.3


## 5. Save the cleaned outputs

The cleaned tables and the merged master table are written to `../data/` with a
`clean_` prefix. The master table joins each line to both of its substations and its
owning utility — it's the table most later questions get answered from, so it's built
once here (details of the join and its checks are in `griddata.build_master`).

In [12]:
master = griddata.build_master(data)
print(f"master table: {master.shape[0]} rows x {master.shape[1]} cols")
print(f"inter-regional lines: {int(master['Inter-regional'].sum())}")
print(f"cross-border lines:  {int(master['Cross-border'].sum())}")

for name, df in data.items():
    df.to_csv(f"../data/clean_{name}.csv", index=False)
master.to_csv("../data/clean_master.csv", index=False)
print("written: clean_utilities.csv, clean_substations.csv, clean_lines.csv, clean_master.csv")

master table: 55 rows x 36 cols
inter-regional lines: 16
cross-border lines:  4
written: clean_utilities.csv, clean_substations.csv, clean_lines.csv, clean_master.csv


## Summary

| check | result |
|---|---|
| rows loaded | 10 utilities / 44 substations / 55 lines |
| missing values | 0 (verified, not assumed) |
| duplicates | 0 |
| broken foreign keys | 0 |
| coordinates out of bounds | 0 |
| isolated substations | 2 — kept, flagged (Conakry Hub, Savelugu) |
| duplicated line pairs | 0 |

The data is fit for analysis. Next: `02-eda.ipynb` for the exploratory questions.